# Temporal Point Process Data Preprocessing

In [1]:
import os
import sys
import json
import pickle
import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

In [2]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [3]:
data_folder = os.path.join('..', 'data')

## Stack Overflow Badges

Download the `stackoverflow.com-Badges.7z` data from [Stack Exchange Data Dump](https://archive.org/details/stackexchange) and unzip it to `data/raw/stack_overflow/Badges.xml`. The data schema can be found from [here](https://meta.stackexchange.com/questions/2677/database-schema-documentation-for-the-public-data-dump-and-sede), and the badge types can be found from [there](https://meta.stackexchange.com/questions/67397/what-are-the-badges-i-can-earn-on-each-site-and-what-are-the-exact-criteria-for).

In [ ]:
!head ../data/raw/stack_overflow/Badges.xml

### Transforming the XML

In [ ]:
import xml.etree.ElementTree as ET
import csv

In [ ]:
def xml_to_csv_chunked(xml_file, csv_file, chunk_size=1000):
    context = ET.iterparse(xml_file, events=('start', 'end'))
    context = iter(context)
    event, root = next(context)
    
    with open(csv_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        header_written = False
        rows = []

        for i, (event, elem) in tqdm(enumerate(context)):
            if event == 'end' and elem.tag == 'row':
                if not header_written:
                    header = list(elem.attrib.keys())
                    writer.writerow(header)
                    header_written = True
                
                rows.append(list(elem.attrib.values()))
                
                if len(rows) >= chunk_size:
                    writer.writerows(rows)
                    rows = []

                root.clear()

        if rows:
            writer.writerows(rows)

In [ ]:
xml_file = f'{data_folder}/raw/stack_overflow/Badges.xml'
csv_file = f'{data_folder}/raw/stack_overflow/badges.csv'
xml_to_csv_chunked(xml_file, csv_file)

### Loading Data

In [ ]:
df_badges = pd.read_csv(f"{data_folder}/raw/stack_overflow/badges.csv")

In [ ]:
df_badges

In [ ]:
df_badges.info()

### Preprocessing Data

In [ ]:
# Drop NaNs and duplicates
df_badges = df_badges.dropna(subset=['UserId', 'Date', 'Name'])\
    .drop_duplicates(subset=['UserId', 'Date', 'Name'], keep='first')

In [ ]:
# Drop tag-affiliated badges
df_badges = df_badges[~df_badges['TagBased']].copy()
# Transform the date times
df_badges['Date'] = pd.to_datetime(df_badges['Date'])

In [ ]:
df_badges['Date'].describe()

In [ ]:
df_badges['Name'].nunique()

In [ ]:
# Select 2-year data
df_badges = df_badges[('2022-01-01' <= df_badges['Date']) & (df_badges['Date'] < '2024-01-01')]

In [ ]:
df_badges['Date'].describe()

### Selecting Badges

Badges:

* Awarded multiple times: Buzz, Socratic, Enlightened, Guru, Lifejacket, Lifeboat, Nice Answer, Good Answer, Great Answer, Populist, Reversal (retired), Revival, Necromancer, Activist, Campaigner, Founder, Good Question, Great Question, Grassroots, Movement, Nice Question, Promoter, Revolution, Steward, Caucus, Constituent, Yearling, Not a Robot

* Awarded once per question: Favorite Question, Stellar Question, Nice Question, Good Question, Great Question, Popular Question, Notable Question, Famous Question

* Awarded once per answer: Favorite Answer, Stellar Answer

* Awarded once per review queue: Custodian, Reviewer

* Awarded once per post: Announcer, Booster, Publicist

In [ ]:
badge_list = [
    "Buzz", "Socratic", "Enlightened", "Guru", "Lifejacket", "Lifeboat",
    "Nice Answer", "Good Answer", "Great Answer", "Populist", "Revival",
    "Necromancer", "Activist", "Campaigner", "Founder", "Good Question",
    "Great Question", "Grassroots", "Movement", "Nice Question", "Promoter",
    "Revolution", "Steward", "Caucus", "Constituent", "Yearling", "Not a Robot",
    "Favorite Question", "Stellar Question", "Nice Question", "Good Question",
    "Great Question", "Popular Question", "Notable Question", "Famous Question",
    "Favorite Answer", "Stellar Answer", "Custodian", "Reviewer", "Announcer",
    "Booster", "Publicist",
]
badge_list = set(badge_list)
len(badge_list)

In [ ]:
# Select badges that can be awarded multiple times
df_badges = df_badges[df_badges["Name"].isin(badge_list)]
len(df_badges)

### Selecting Users

In [ ]:
user_badge_counts = df_badges.groupby('UserId')['Name'].count()

In [ ]:
user_badge_counts.describe().astype(int)

In [ ]:
# Select users who have earned at 40-100 badges
user_list = user_badge_counts[(user_badge_counts >= 40) & (user_badge_counts <= 100)].index
len(user_list)

In [ ]:
df_badges = df_badges[df_badges["UserId"].isin(user_list)]
len(df_badges)

### Selecting Badges Again

In [ ]:
badge_type_counts = df_badges['Name'].value_counts()

In [ ]:
# Select badges which have been awarded at least 200 times
badge_type_list = badge_type_counts[badge_type_counts >= 200].index
len(badge_type_list)

In [ ]:
df_badges = df_badges[df_badges["Name"].isin(badge_type_list)]
len(df_badges)

### Splitting Sequences

In [ ]:
df_badges.groupby('UserId')['Date'].count().describe()

In [ ]:
df_badges['Name'].value_counts()

In [26]:
def get_seq_splits(df, seq_col):
    seq_ids = df[seq_col].unique().tolist()
    seq_ids_train, seq_ids_val_test = train_test_split(seq_ids, train_size=0.8, random_state=0)
    seq_ids_val, seq_ids_test = train_test_split(seq_ids_val_test, train_size=0.5, random_state=0)
    seq_splits = {seq_id: 'train' for seq_id in seq_ids_train}
    seq_splits.update({seq_id: 'dev' for seq_id in seq_ids_val})
    seq_splits.update({seq_id: 'test' for seq_id in seq_ids_test})
    print(f'train: {len(seq_ids_train)} seqs, val: {len(seq_ids_val)} seqs, test: {len(seq_ids_test)} seqs')
    return seq_splits

In [ ]:
badge_seq_splits = get_seq_splits(df=df_badges, seq_col='UserId')
len(badge_seq_splits)

### Saving Sequences

In [28]:
def save_seqs(
    df: pd.DataFrame, seq_col: str, seq_splits: dict,
    time_col: str, time_unit: float, type_col: str, seq_folder: str):
    """
    Save event sequences
    """
    dim_process = df[type_col].nunique()
    type_text2id = {type_text: type_id for type_id, type_text in enumerate(df[type_col].unique())}
    type_id2text = {type_id: type_text for type_text, type_id in type_text2id.items()}
    type_id_col = f'{type_col}_id'
    df[type_id_col] = df[type_col].map(type_text2id)
    data = {'train': [], 'dev': [], 'test': []}
    print(f'type_id2text: {type_id2text}')
    
    for seq_id, group in tqdm(df.groupby(seq_col)):
        group = group.sort_values(by=time_col).reset_index()
        split = seq_splits[seq_id]
        init_time = group[time_col].min()
        pre_event_time = init_time
        event_seq = {
            'dim_process': dim_process,
            'seq_idx': len(data[split]),
            'seq_len': len(group),
            'time_since_start': [],
            'time_since_last_event': [],
            'type_event': [],
            'type_text': [],
        }
        
        for index, row in group.iterrows():
            event_time = pd.to_datetime(row[time_col])
            time_since_start = (event_time - init_time).total_seconds() / time_unit
            time_since_last_event = (event_time - pre_event_time).total_seconds() / time_unit
            event_seq['time_since_start'].append(time_since_start)
            event_seq['time_since_last_event'].append(time_since_last_event)
            event_seq['type_event'].append(row[type_id_col])
            event_seq['type_text'].append(row[type_col])
            pre_event_time = event_time
        
        data[split].append(event_seq)

    os.makedirs(seq_folder, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        json_path = f'{seq_folder}/{split}.json'
        with open(json_path, 'w') as file:
            json.dump(data[split], file, indent=4)
        print(f'{split} saved to {json_path}')

In [ ]:
save_seqs(
    df=df_badges, seq_col='UserId', seq_splits=badge_seq_splits,
    time_col='Date', time_unit=60*60*24*30, type_col='Name',
    seq_folder=f'{data_folder}/stack_overflow',
)

## Chicago Crimes

Download the data from [Crimes - 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data) to `data/raw/chicago_crime/Crimes_-_2001_to_Present.csv`.

### Loading Data

In [5]:
df_crimes = pd.read_csv("data/raw/chicago_crime/Crimes_-_2001_to_Present_20250809.csv")

NameError: name 'pd' is not defined

In [40]:
df_crimes

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,13919562,JJ356542,07/30/2025 12:00:00 AM,005XX W ARLINGTON PL,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,1935,19.0,43.0,7.0,14,1172332.0,1916661.0,2025,08/06/2025 03:47:35 PM,41.926765,-87.642173,"(41.926764863, -87.642172789)"
1,13924375,JJ356066,07/30/2025 12:00:00 AM,090XX S BRANDON AVE,0560,ASSAULT,SIMPLE,RESIDENCE,False,True,424,4.0,10.0,46.0,08A,NaN,NaN,2025,08/06/2025 03:47:35 PM,NaN,NaN,NaN
2,13917551,JJ354466,07/30/2025 12:00:00 AM,049XX W MONROE ST,1310,CRIMINAL DAMAGE,TO PROPERTY,APARTMENT,False,True,1533,15.0,28.0,25.0,14,1143545.0,1899209.0,2025,08/06/2025 03:47:35 PM,41.879463,-87.748390,"(41.879462649, -87.748390237)"
3,13918531,JJ355633,07/30/2025 12:00:00 AM,019XX W CERMAK RD,0820,THEFT,$500 AND UNDER,SMALL RETAIL STORE,True,False,1034,10.0,25.0,31.0,06,1163777.0,1889420.0,2025,08/06/2025 03:47:35 PM,41.852198,-87.674377,"(41.852198325, -87.674377204)"
4,13921703,JJ359374,07/30/2025 12:00:00 AM,021XX W FARRAGUT AVE,0810,THEFT,OVER $500,STREET,False,False,2012,20.0,40.0,4.0,06,1161169.0,1934842.0,2025,08/06/2025 03:47:35 PM,41.976894,-87.682685,"(41.976894099, -87.682684568)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8373657,2619446,HJ222971,01/01/2001 12:00:00 AM,022XX S CENTRAL PARK AVE,1754,OFFENSE INVOLVING CHILDREN,AGG SEX ASSLT OF CHILD FAM MBR,RESIDENCE,False,False,1013,10.0,22.0,30.0,02,1152685.0,1888677.0,2001,08/17/2015 03:03:40 PM,41.850386,-87.715108,"(41.850385805, -87.715107802)"
8373658,2980108,HJ677840,01/01/2001 12:00:00 AM,045XX N MC VICKER AVE,0840,THEFT,FINANCIAL ID THEFT: OVER $300,RESIDENCE,False,False,1622,16.0,38.0,15.0,06,1135231.0,1929683.0,2001,03/31/2006 10:03:38 PM,41.963238,-87.778194,"(41.96323831, -87.778194071)"
8373659,2353946,HH655999,01/01/2001 12:00:00 AM,044XX S CALIFORNIA AVE,0841,THEFT,FINANCIAL ID THEFT:$300 &UNDER,RESIDENCE,False,False,912,9.0,12.0,58.0,06,1158408.0,1874895.0,2001,08/17/2015 03:03:40 PM,41.812451,-87.694479,"(41.812451386, -87.694479455)"
8373660,2893567,HJ560176,01/01/2001 12:00:00 AM,087XX S ELIZABETH ST,0840,THEFT,FINANCIAL ID THEFT: OVER $300,RESIDENCE,False,False,2222,22.0,21.0,71.0,06,1169564.0,1846653.0,2001,03/31/2006 10:03:38 PM,41.734717,-87.654377,"(41.734716977, -87.654377051)"


In [41]:
df_crimes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8373662 entries, 0 to 8373661
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   ID                    int64  
 1   Case Number           object 
 2   Date                  object 
 3   Block                 object 
 4   IUCR                  object 
 5   Primary Type          object 
 6   Description           object 
 7   Location Description  object 
 8   Arrest                bool   
 9   Domestic              bool   
 10  Beat                  int64  
 11  District              float64
 12  Ward                  float64
 13  Community Area        float64
 14  FBI Code              object 
 15  X Coordinate          float64
 16  Y Coordinate          float64
 17  Year                  int64  
 18  Updated On            object 
 19  Latitude              float64
 20  Longitude             float64
 21  Location              object 
dtypes: bool(2), float64(7), int64(3), object(1

In [3]:
df_crimes

NameError: name 'df_crimes' is not defined

In [43]:
df_crimes.isna().sum()

ID                           0
Case Number                  0
Date                         0
Block                        0
IUCR                         0
Primary Type                 0
Description                  0
Location Description     14636
Arrest                       0
Domestic                     0
Beat                         0
District                    47
Ward                    614823
Community Area          613688
FBI Code                     0
X Coordinate             93400
Y Coordinate             93400
Year                         0
Updated On                   0
Latitude                 93400
Longitude                93400
Location                 93400
dtype: int64

### Preprocessing Data

In [44]:
df_crimes['Date'] = pd.to_datetime(df_crimes['Date'])
df_crimes['Primary Type'] = df_crimes['Primary Type'].str.title()

C:\Users\akech\AppData\Local\Temp\ipykernel_6432\1849376.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_crimes['Date'] = pd.to_datetime(df_crimes['Date'])


In [45]:
df_crimes = df_crimes.dropna(subset=['Date', 'Block', 'Primary Type','Latitude','Longitude'])\
    .drop_duplicates(subset=['Date', 'Block', 'Primary Type','Latitude','Longitude'], keep='first')

In [46]:
df_crimes = df_crimes[('2022-01-01' <= df_crimes['Date']) & (df_crimes['Date'] < '2024-01-01')]
len(df_crimes)

494769

### Selecting Crimes

In [47]:
crime_counts = df_crimes['Primary Type'].value_counts()

In [48]:
crime_counts

Primary Type
Theft                                110324
Battery                               84750
Criminal Damage                       56863
Motor Vehicle Theft                   50429
Assault                               43230
Deceptive Practice                    32124
Other Offense                         29862
Robbery                               19966
Weapons Violation                     17291
Burglary                              14969
Narcotics                              9245
Criminal Trespass                      8896
Offense Involving Children             3416
Criminal Sexual Assault                3094
Sex Offense                            2457
Public Peace Violation                 1566
Homicide                               1359
Interference With Public Officer        977
Stalking                                950
Arson                                   933
Prostitution                            487
Intimidation                            412
Liquor Law Violatio

In [49]:
crime_list = crime_counts[crime_counts >= 500].index
len(crime_list)

20

In [50]:
df_crimes = df_crimes[df_crimes['Primary Type'].isin(crime_list)]
len(df_crimes)

492701

In [8]:
df_crimes

NameError: name 'df_crimes' is not defined

### Selecting Blocks

In [51]:
block_counts = df_crimes['Block'].value_counts()

In [52]:
block_counts

Block
001XX N STATE ST        1430
0000X W TERMINAL ST     1018
0000X N STATE ST         711
100XX W OHARE ST         542
0000X E GRAND AVE        529
                        ... 
006XX W 63rd st            1
026XX W Iowa st            1
039XX W 45TH ST            1
061XX N LOWELL AVE         1
037XX S CAMPBELL AVE       1
Name: count, Length: 31725, dtype: int64

In [7]:
block_list = block_counts[(30 <= block_counts) & (block_counts <= 120)].index
len(block_list)

NameError: name 'block_counts' is not defined

In [54]:
df_crimes = df_crimes[df_crimes['Block'].isin(block_list)]
len(df_crimes)

199623

In [55]:
df_crimes['Primary Type'].nunique()

20

In [56]:
df_crimes['Primary Type'].value_counts()

Primary Type
Theft                               41767
Battery                             39266
Criminal Damage                     22423
Assault                             19843
Motor Vehicle Theft                 17773
Other Offense                       11853
Deceptive Practice                  11293
Robbery                              8318
Weapons Violation                    6911
Burglary                             5508
Narcotics                            4875
Criminal Trespass                    3609
Criminal Sexual Assault              1368
Offense Involving Children           1340
Sex Offense                          1018
Public Peace Violation                699
Homicide                              567
Interference With Public Officer      437
Stalking                              395
Arson                                 360
Name: count, dtype: int64

### Saving Sequences

In [57]:
df_crimes.groupby('Block')['Date'].count().describe()

count    3983.000000
mean       50.118755
std        20.355127
min        30.000000
25%        35.000000
50%        43.000000
75%        59.000000
max       120.000000
Name: Date, dtype: float64

In [58]:
crime_seq_splits = get_seq_splits(df=df_crimes, seq_col='Block')
len(crime_seq_splits)

train: 3186 seqs, val: 398 seqs, test: 399 seqs


3983

In [2]:
df_crimes

NameError: name 'df_crimes' is not defined

In [59]:
def save_seqs(
    df: pd.DataFrame, seq_col: str, seq_splits: dict,
    time_col: str, time_unit: float, type_col: str,lat_col: str,lon_col: str, seq_folder: str):
    """
    Save event sequences
    """
    dim_process = df[type_col].nunique()
    type_text2id = {type_text: type_id for type_id, type_text in enumerate(df[type_col].unique())}
    type_id2text = {type_id: type_text for type_text, type_id in type_text2id.items()}
    type_id_col = f'{type_col}_id'
    df[type_id_col] = df[type_col].map(type_text2id)
    data = {'train': [], 'dev': [], 'test': []}
    print(f'type_id2text: {type_id2text}')
    
    for seq_id, group in tqdm(df.groupby(seq_col)):
        group = group.sort_values(by=time_col).reset_index()
        split = seq_splits[seq_id]
        init_time = group[time_col].min()
        pre_event_time = init_time
        event_seq = {
            'dim_process': dim_process,
            'seq_idx': len(data[split]),
            'seq_len': len(group),
            'time_since_start': [],
            'time_since_last_event': [],
            'type_event': [],
            'type_text': [],
            'Latitude': [],
            'Longitude': [],
        }
        
        for index, row in group.iterrows():
            event_time = pd.to_datetime(row[time_col])
            time_since_start = (event_time - init_time).total_seconds() / time_unit
            time_since_last_event = (event_time - pre_event_time).total_seconds() / time_unit
            event_seq['time_since_start'].append(time_since_start)
            event_seq['time_since_last_event'].append(time_since_last_event)
            event_seq['type_event'].append(row[type_id_col])
            event_seq['type_text'].append(row[type_col])
            event_seq['Latitude'].append(row[lat_col])
            event_seq['Longitude'].append(row[lon_col])
            pre_event_time = event_time
        
        data[split].append(event_seq)

    os.makedirs(seq_folder, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        json_path = f'{seq_folder}/{split}.json'
        with open(json_path, 'w') as file:
            json.dump(data[split], file, indent=4)
        print(f'{split} saved to {json_path}')

In [60]:
save_seqs(
    df=df_crimes, seq_col='Block', seq_splits=crime_seq_splits,
    time_col='Date', time_unit=60*60*24*30, type_col='Primary Type',lat_col='Latitude',lon_col='Longitude',
    seq_folder=f'{data_folder}/chicago_crime',
)

type_id2text: {0: 'Battery', 1: 'Theft', 2: 'Public Peace Violation', 3: 'Criminal Damage', 4: 'Robbery', 5: 'Other Offense', 6: 'Weapons Violation', 7: 'Criminal Trespass', 8: 'Motor Vehicle Theft', 9: 'Assault', 10: 'Stalking', 11: 'Burglary', 12: 'Deceptive Practice', 13: 'Arson', 14: 'Interference With Public Officer', 15: 'Narcotics', 16: 'Offense Involving Children', 17: 'Sex Offense', 18: 'Criminal Sexual Assault', 19: 'Homicide'}


  0%|          | 0/3983 [00:00<?, ?it/s]

train saved to ..\data/chicago_crime/train.json
dev saved to ..\data/chicago_crime/dev.json
test saved to ..\data/chicago_crime/test.json


## NYC Taxi Trips

Download the [NYC Taxi Trips](https://www.andresmh.com/nyctaxitrips/) to `data/raw/nyc_taxi/` and [NYC Borough Boundaries](https://data.cityofnewyork.us/City-Government/Borough-Boundaries/tqmj-j8zm) ("Export" then "Original") to `data/raw/nyc_taxi/nybb_24c/`.

In [23]:
pip install sodapy

Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
from sodapy import Socrata
import time
from datetime import datetime, timedelta
import requests

# ===== 設定 =====
DATASET_ID = "t7ny-aygi"    # データセットID
APP_TOKEN = None            # app_tokenがあれば "xxxxx" に設定
TIMEOUT = 60                # タイムアウト秒数
LIMIT = 2000                # 1回の取得件数
RETRY_MAX = 3               # リトライ最大回数
SLEEP_BETWEEN_REQUESTS = 0.5 # API負荷軽減のための休憩秒数
START_DATE_STR = "2013-05-01"
END_DATE_STR   = "2013-05-07"
# =================

# Socrataクライアント作成
client = Socrata("data.cityofnewyork.us", APP_TOKEN, timeout=TIMEOUT)

# 日付カラム自動検出
metadata = client.get_metadata(DATASET_ID)
datetime_columns = [
    col['fieldName'] for col in metadata['columns']
    if col.get('dataTypeName') in ('calendar_date', 'date', 'calendar_date_time', 'floating_timestamp')
]
if not datetime_columns:
    raise ValueError("日付/日時型カラムが見つかりません")
date_column = datetime_columns[0]  # 最初の候補を使う
print(f"検出した日付カラム: {date_column}")

# 取得範囲
start_date = datetime.strptime(START_DATE_STR, "%Y-%m-%d")
end_date   = datetime.strptime(END_DATE_STR, "%Y-%m-%d")

all_results = []
current_start = start_date

# 半日ごとに分割
while current_start <= end_date:
    current_end = min(current_start + timedelta(hours=12) - timedelta(seconds=1), end_date + timedelta(days=1) - timedelta(seconds=1))
    day_start_str = current_start.strftime("%Y-%m-%dT%H:%M:%S")
    day_end_str   = current_end.strftime("%Y-%m-%dT%H:%M:%S")

    print(f"=== {day_start_str} 〜 {day_end_str} ===")

    offset = 0
    while True:
        # リトライ付き取得
        for attempt in range(RETRY_MAX):
            try:
                results = client.get(
                    DATASET_ID,
                    where=f"{date_column} between '{day_start_str}' and '{day_end_str}'",
                    limit=LIMIT,
                    offset=offset
                )
                break  # 成功したらループ脱出
            except (requests.exceptions.RequestException, requests.exceptions.HTTPError) as e:
                print(f"  エラー発生（試行 {attempt+1}/{RETRY_MAX}）: {e}")
                time.sleep(2)
        else:
            print("  最大リトライ回数を超えたためスキップ")
            break

        if not results:
            break

        all_results.extend(results)
        offset += LIMIT
        print(f"  {len(results)} 件取得（累計 {len(all_results)} 件）")
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    current_start += timedelta(hours=12)

# DataFrame化
results_df = pd.DataFrame.from_records(all_results)
print(f"\n【最終取得件数】 {len(results_df)} 件")
print(results_df.head())

# CSV保存
filename = f"nyc_data_{START_DATE_STR}_{END_DATE_STR}.csv"
results_df.to_csv(filename, index=False)
print(f"CSVに保存しました: {filename}")

検出した日付カラム: tpep_pickup_datetime
=== 2013-05-01T00:00:00 〜 2013-05-01T11:59:59 ===
  2000 件取得（累計 2000 件）
  2000 件取得（累計 4000 件）
  エラー発生（試行 1/3）: 500 Server Error: Server Error.
	Internal error: please include code 509a41f1-b595-4089-ac51-304d1e8a8d30 if you report the error
  2000 件取得（累計 6000 件）
  エラー発生（試行 1/3）: 500 Server Error: Server Error.
	Internal error: please include code 00007686-f375-40ae-bc29-7b725866f8dd if you report the error
  エラー発生（試行 2/3）: 500 Server Error: Server Error.
	Internal error: please include code d753a29c-8266-4403-b8c2-15ceace98229 if you report the error
  2000 件取得（累計 8000 件）
  2000 件取得（累計 10000 件）
  2000 件取得（累計 12000 件）
  2000 件取得（累計 14000 件）
  2000 件取得（累計 16000 件）
  2000 件取得（累計 18000 件）
  2000 件取得（累計 20000 件）
  2000 件取得（累計 22000 件）
  2000 件取得（累計 24000 件）
  2000 件取得（累計 26000 件）
  2000 件取得（累計 28000 件）
  2000 件取得（累計 30000 件）
  2000 件取得（累計 32000 件）
  2000 件取得（累計 34000 件）
  2000 件取得（累計 36000 件）
  2000 件取得（累計 38000 件）
  2000 件取得（累計 40000 件）
  2000 件取得（累計 42000 件

KeyboardInterrupt: 

In [6]:
import os
import pandas as pd
from sodapy import Socrata
import time
from datetime import datetime, timedelta
import requests

# ===== 設定 =====
DATASET_ID = "t7ny-aygi"    # データセットID
APP_TOKEN = None            # app_tokenがあれば "xxxxx" に設定
TIMEOUT = 60
LIMIT = 2000
RETRY_MAX = 3
SLEEP_BETWEEN_REQUESTS = 0.5
START_DATE_STR = "2013-05-01"
END_DATE_STR   = "2013-05-08"
OUTPUT_DIR = "nyc_data_parts"
# =================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Socrataクライアント作成
client = Socrata("data.cityofnewyork.us", APP_TOKEN, timeout=TIMEOUT)

# 日付カラム自動検出
metadata = client.get_metadata(DATASET_ID)
datetime_columns = [
    col['fieldName'] for col in metadata['columns']
    if col.get('dataTypeName') in ('calendar_date', 'date', 'calendar_date_time', 'floating_timestamp')
]
if not datetime_columns:
    raise ValueError("日付/日時型カラムが見つかりません")
date_column = datetime_columns[0]
print(f"検出した日付カラム: {date_column}")

# 取得範囲
start_date = datetime.strptime(START_DATE_STR, "%Y-%m-%d")
end_date   = datetime.strptime(END_DATE_STR, "%Y-%m-%d")

current_start = start_date

# 半日ごとに分割
while current_start <= end_date:
    current_end = min(current_start + timedelta(hours=12) - timedelta(seconds=1),
                      end_date + timedelta(days=1) - timedelta(seconds=1))
    day_start_str = current_start.strftime("%Y-%m-%dT%H:%M:%S")
    day_end_str   = current_end.strftime("%Y-%m-%dT%H:%M:%S")

    # 保存ファイル名（日付時間をファイル名に含める）
    filename = os.path.join(OUTPUT_DIR, f"{day_start_str.replace(':', '-')}_to_{day_end_str.replace(':', '-')}.csv")

    # すでに取得済みならスキップ
    if os.path.exists(filename):
        print(f"=== {day_start_str} 〜 {day_end_str} ===  → 保存済みスキップ")
        current_start += timedelta(hours=12)
        continue

    print(f"=== {day_start_str} 〜 {day_end_str} ===")

    results_for_period = []
    offset = 0
    while True:
        # リトライ付き取得
        for attempt in range(RETRY_MAX):
            try:
                results = client.get(
                    DATASET_ID,
                    where=f"{date_column} between '{day_start_str}' and '{day_end_str}'",
                    limit=LIMIT,
                    offset=offset
                )
                break
            except (requests.exceptions.RequestException, requests.exceptions.HTTPError) as e:
                print(f"  エラー発生（試行 {attempt+1}/{RETRY_MAX}）: {e}")
                time.sleep(2)
        else:
            print("  最大リトライ回数を超えたためスキップ")
            break

        if not results:
            break

        results_for_period.extend(results)
        offset += LIMIT
        print(f"  {len(results)} 件取得（期間累計 {len(results_for_period)} 件）")
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    # その期間のデータを保存
    if results_for_period:
        df = pd.DataFrame.from_records(results_for_period)
        df.to_csv(filename, index=False)
        print(f"  保存完了: {filename}")
    else:
        print("  データなし")

    current_start += timedelta(hours=12)

print("\nすべての期間の処理が完了しました。")


検出した日付カラム: tpep_pickup_datetime
=== 2013-05-01T00:00:00 〜 2013-05-01T11:59:59 ===  → 保存済みスキップ
=== 2013-05-01T12:00:00 〜 2013-05-01T23:59:59 ===  → 保存済みスキップ
=== 2013-05-02T00:00:00 〜 2013-05-02T11:59:59 ===  → 保存済みスキップ
=== 2013-05-02T12:00:00 〜 2013-05-02T23:59:59 ===  → 保存済みスキップ
=== 2013-05-03T00:00:00 〜 2013-05-03T11:59:59 ===  → 保存済みスキップ
=== 2013-05-03T12:00:00 〜 2013-05-03T23:59:59 ===  → 保存済みスキップ
=== 2013-05-04T00:00:00 〜 2013-05-04T11:59:59 ===
  2000 件取得（期間累計 2000 件）
  2000 件取得（期間累計 4000 件）
  2000 件取得（期間累計 6000 件）
  2000 件取得（期間累計 8000 件）
  2000 件取得（期間累計 10000 件）
  2000 件取得（期間累計 12000 件）
  2000 件取得（期間累計 14000 件）
  2000 件取得（期間累計 16000 件）
  2000 件取得（期間累計 18000 件）
  2000 件取得（期間累計 20000 件）
  2000 件取得（期間累計 22000 件）
  2000 件取得（期間累計 24000 件）
  2000 件取得（期間累計 26000 件）
  2000 件取得（期間累計 28000 件）
  2000 件取得（期間累計 30000 件）
  2000 件取得（期間累計 32000 件）
  2000 件取得（期間累計 34000 件）
  2000 件取得（期間累計 36000 件）
  2000 件取得（期間累計 38000 件）
  2000 件取得（期間累計 40000 件）
  2000 件取得（期間累計 42000 件）
  2000 件取得（期間累計 44000 件）


In [ ]:
2013-05-03T12-00-00_to_2013-05-03T23-59-59

In [24]:
import pandas as pd
from sodapy import Socrata

# Unauthenticated client only works with public data sets. Note 'None'
# in place of application token, and no username or password:
client = Socrata("data.cityofnewyork.us", None)

# Example authenticated client (needed for non-public datasets):
# client = Socrata(data.cityofnewyork.us,
#                  MyAppToken,
#                  username="user@example.com",
#                  password="AFakePassword")

# First 2000 results, returned as JSON from API / converted to Python list of
# dictionaries by sodapy.
results = client.get("t7ny-aygi", limit=2000)

# Convert to pandas DataFrame
results_df = pd.DataFrame.from_records(results)

In [25]:
results_df

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,ratecodeid,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount,pickup_location,dropoff_location,store_and_fwd_flag
0,VTS,2013-06-11T22:41:00.000,2013-06-11T22:55:00.000,3,2.52,-73.982020000000006,40.771436999999999,1,-73.947145000000006,40.771791999999998,CSH,12,0.5,0.5,0,0,13,"{'type': 'Point', 'coordinates': [-73.98202, 4...","{'type': 'Point', 'coordinates': [-73.947145, ...",NaN
1,CMT,2013-10-31T17:27:31.000,2013-10-31T17:44:43.000,1,1.3,-73.992648000000003,40.749465000000001,1,-73.981876999999997,40.758702999999997,CSH,11.5,1,0.5,0,0,13,"{'type': 'Point', 'coordinates': [-73.992648, ...","{'type': 'Point', 'coordinates': [-73.981877, ...",N
2,CMT,2013-05-03T11:38:40.000,2013-05-03T11:59:18.000,1,2.6000000000000001,-74.006511000000003,40.751148999999998,1,-73.993405999999993,40.727930999999998,CRD,14,0,0.5,1,0,15.5,"{'type': 'Point', 'coordinates': [-74.006511, ...","{'type': 'Point', 'coordinates': [-73.993406, ...",N
3,VTS,2013-06-21T10:02:00.000,2013-06-21T10:06:00.000,6,0.42999999999999999,-73.992795000000001,40.733992000000001,1,-73.990031999999999,40.731946999999998,CSH,4.5,0,0.5,0,0,5,"{'type': 'Point', 'coordinates': [-73.992795, ...","{'type': 'Point', 'coordinates': [-73.990032, ...",NaN
4,CMT,2013-01-12T18:59:23.000,2013-01-12T19:16:19.000,1,7.0999999999999996,-73.976201000000003,40.781095000000001,1,-73.921153000000004,40.842058999999999,CRD,22.5,0,0.5,0,0,23,"{'type': 'Point', 'coordinates': [-73.976201, ...","{'type': 'Point', 'coordinates': [-73.921153, ...",N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,CMT,2013-08-22T23:43:03.000,2013-08-23T00:02:22.000,1,5.7999999999999998,-73.991017999999997,40.743569000000001,1,-73.940758000000002,40.720230999999998,CRD,20,0.5,0.5,2,0,23,"{'type': 'Point', 'coordinates': [-73.991018, ...","{'type': 'Point', 'coordinates': [-73.940758, ...",N
1996,CMT,2013-08-02T16:22:11.000,2013-08-02T16:33:09.000,1,1.1000000000000001,-73.990826999999996,40.734636999999999,1,-73.987071999999998,40.722543000000002,CSH,8,1,0.5,0,0,9.5,"{'type': 'Point', 'coordinates': [-73.990827, ...","{'type': 'Point', 'coordinates': [-73.987072, ...",N
1997,VTS,2013-10-13T04:53:00.000,2013-10-13T04:56:00.000,6,0.65000000000000002,-74.004746999999995,40.737591999999999,1,-73.985145000000003,40.732537000000001,CRD,4.5,0.5,0.5,1,0,6.5,"{'type': 'Point', 'coordinates': [-74.004747, ...","{'type': 'Point', 'coordinates': [-73.985145, ...",NaN
1998,CMT,2013-05-27T19:10:04.000,2013-05-27T19:19:45.000,1,1.6000000000000001,-74.000238999999993,40.726761000000003,1,-73.978317000000004,40.734319999999997,CRD,8.5,0,0.5,1,0,10,"{'type': 'Point', 'coordinates': [-74.000239, ...","{'type': 'Point', 'coordinates': [-73.978317, ...",N


In [18]:
import zipfile
import os

zip_path = "data/raw/nyc_taxi/trip_data.7z"  # ZIP ファイルのパス
extract_folder = "data/raw/nyc_taxi"  # 展開先フォルダ

# 展開先フォルダが存在しなければ作成
os.makedirs(extract_folder, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    print("展開完了")
except zipfile.BadZipFile:
    print("ZIP ファイルが壊れている可能性があります")
except Exception as e:
    print(f"エラー発生: {e}")

ZIP ファイルが壊れている可能性があります


In [21]:
pip install py7zr


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 32.8 MB/s eta 0:00:00
  Attempting uninstall: brotli
    Found existing installation: Brotli 1.0.9
    Uninstalling Brotli-1.0.9:
      Successfully uninstalled Brotli-1.0.9
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0


In [22]:
import py7zr
import os

archive_path = "data/raw/nyc_taxi/trip_data.7z"  # 7z ファイル
extract_folder = "data/raw/nyc_taxi"  # 展開先

os.makedirs(extract_folder, exist_ok=True)

try:
    with py7zr.SevenZipFile(archive_path, mode='r') as archive:
        archive.extractall(path=extract_folder)
    print("展開完了")
except py7zr.exceptions.Bad7zFile:
    print("7z ファイルが壊れている可能性があります")
except Exception as e:
    print(f"エラー発生: {e}")

展開完了


In [19]:
import zipfile

zip_path = "data/raw/nyc_taxi/trip_data.7z"
extract_folder = "data/raw/nyc_taxi"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for member in zip_ref.namelist():
        try:
            zip_ref.extract(member, extract_folder)
            print(f"展開成功: {member}")
        except:
            print(f"壊れている可能性あり: {member}")

BadZipFile: File is not a zip file

### Loading Data

In [12]:


#!/usr/bin/python
# -*- coding:utf-8 -*-

import sys
import pandas

if __name__ == '__main__':
    args = sys.argv

    df = pandas.read_parquet("data/raw/nyc_taxi/yellow_tripdata_2013-05.parquet")
   # index=Falseで左端の行番号を非表示にできる
    df.to_csv("data/raw/nyc_taxi/yellow_tripdata_2013-05.csv",index=False)

In [37]:
import pandas as pd
import glob
import os

# フォルダパス
folder_path = "data/raw/nyc_taxi"

# フォルダ内の CSV ファイル一覧を取得
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# 各 CSV を読み込んで結合
df_list = [pd.read_csv(file) for file in csv_files]
df_all = pd.concat(df_list, ignore_index=True)

print(f"{len(csv_files)} 個のファイルを読み込みました。")
print(df_all.head())

C:\Users\akech\AppData\Local\Temp\ipykernel_23724\1347032701.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in csv_files]
C:\Users\akech\AppData\Local\Temp\ipykernel_23724\1347032701.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in csv_files]
C:\Users\akech\AppData\Local\Temp\ipykernel_23724\1347032701.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in csv_files]
C:\Users\akech\AppData\Local\Temp\ipykernel_23724\1347032701.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in csv_files]
C:\Users\akech\AppData\Local\Temp\ipykernel_23724\1347032701.py:12: DtypeWarning: Columns (4) have mixed types. 

MemoryError: Unable to allocate 6.45 GiB for an array with shape (5, 173179759) and data type float64

In [4]:
df_trips = pd.read_csv("data/raw/nyc_taxi/trip_data_5.csv")

C:\Users\akech\AppData\Local\Temp\ipykernel_13648\1970854806.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_trips = pd.read_csv("data/raw/nyc_taxi/trip_data_5.csv")


In [ ]:
df_trips=df_all

In [5]:
df_trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15285049 entries, 0 to 15285048
Data columns (total 14 columns):
 #   Column               Dtype  
---  ------               -----  
 0   medallion            object 
 1    hack_license        object 
 2    vendor_id           object 
 3    rate_code           int64  
 4    store_and_fwd_flag  object 
 5    pickup_datetime     object 
 6    dropoff_datetime    object 
 7    passenger_count     int64  
 8    trip_time_in_secs   int64  
 9    trip_distance       float64
 10   pickup_longitude    float64
 11   pickup_latitude     float64
 12   dropoff_longitude   float64
 13   dropoff_latitude    float64
dtypes: float64(5), int64(3), object(6)
memory usage: 1.6+ GB


In [ ]:
df_trips.info()

In [ ]:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12748986 entries, 0 to 12748985
Data columns (total 19 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        int64  
 4   trip_distance          float64
 5   pickup_longitude       float64
 6   pickup_latitude        float64
 7   RateCodeID             int64  
 8   store_and_fwd_flag     object 
 9   dropoff_longitude      float64
 10  dropoff_latitude       float64
 11  payment_type           int64  
 12  fare_amount            float64
 13  extra                  float64
 14  mta_tax                float64
 15  tip_amount             float64
 16  tolls_amount           float64
 17  improvement_surcharge  float64
 18  total_amount           float64
dtypes: float64(12), int64(4), object(3)
memory usage: 1.8+ GB

In [9]:
df_trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12748986 entries, 0 to 12748985
Data columns (total 19 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        int64  
 4   trip_distance          float64
 5   pickup_longitude       float64
 6   pickup_latitude        float64
 7   RateCodeID             int64  
 8   store_and_fwd_flag     object 
 9   dropoff_longitude      float64
 10  dropoff_latitude       float64
 11  payment_type           int64  
 12  fare_amount            float64
 13  extra                  float64
 14  mta_tax                float64
 15  tip_amount             float64
 16  tolls_amount           float64
 17  improvement_surcharge  float64
 18  total_amount           float64
dtypes: float64(12), int64(4), object(3)
memory usage: 1.8+ GB


In [16]:
df_trips_table = pd.read_csv("data/raw/nyc_taxi/taxi_zone_lookup.csv")

In [21]:
df_trips_table[df_trips_table['LocationID']==79]

,LocationID,Borough,Zone,service_zone
78,79,Manhattan,East Village,Yellow Zone


In [11]:
df_all

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,ratecodeid,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount,pickup_location,dropoff_location,store_and_fwd_flag
0,VTS,2013-05-01T00:00:00.000,2013-05-01T00:03:00.000,1,0.61,-73.953387,40.728452,1,-73.958410,40.731997,CSH,4.5,0.5,0.5,0.0,0.0,5.5,"{'type': 'Point', 'coordinates': [-73.953387, ...","{'type': 'Point', 'coordinates': [-73.95841, 4...",NaN
1,CMT,2013-05-01T00:00:00.000,2013-05-01T00:10:08.000,1,2.50,-73.999056,40.732460,1,-73.975808,40.756539,CSH,9.5,0.5,0.5,0.0,0.0,10.5,"{'type': 'Point', 'coordinates': [-73.999056, ...","{'type': 'Point', 'coordinates': [-73.975808, ...",N
2,VTS,2013-05-01T00:00:00.000,2013-05-01T00:08:00.000,1,1.91,-73.986187,40.722815,1,-73.977335,40.742807,CSH,8.5,0.5,0.5,0.0,0.0,9.5,"{'type': 'Point', 'coordinates': [-73.986187, ...","{'type': 'Point', 'coordinates': [-73.977335, ...",NaN
3,VTS,2013-05-01T00:00:00.000,2013-05-01T00:08:00.000,5,2.32,-73.980902,40.761555,1,-73.953105,40.771952,CRD,9.5,0.5,0.5,2.0,0.0,12.5,"{'type': 'Point', 'coordinates': [-73.980902, ...","{'type': 'Point', 'coordinates': [-73.953105, ...",NaN
4,VTS,2013-05-01T00:00:00.000,2013-05-01T00:16:00.000,2,3.96,-73.961618,40.719287,1,-74.004275,40.722832,CSH,15.5,0.5,0.5,0.0,0.0,16.5,"{'type': 'Point', 'coordinates': [-73.961618, ...","{'type': 'Point', 'coordinates': [-74.004275, ...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3759337,CMT,2013-05-08T01:08:58.000,2013-05-08T01:21:20.000,1,3.20,-73.975287,40.762611,1,-74.001110,40.721905,CSH,12.0,0.5,0.5,0.0,0.0,13.0,"{'type': 'Point', 'coordinates': [-73.975287, ...","{'type': 'Point', 'coordinates': [-74.00111, 4...",N
3759338,VTS,2013-05-08T09:32:00.000,2013-05-08T09:36:00.000,6,0.79,-73.960032,40.801372,1,-73.977543,40.787037,CRD,5.5,0.0,0.5,0.0,0.0,6.0,"{'type': 'Point', 'coordinates': [-73.960032, ...","{'type': 'Point', 'coordinates': [-73.977543, ...",NaN
3759339,VTS,2013-05-08T07:34:00.000,2013-05-08T08:19:00.000,1,6.67,-73.963672,40.762020,1,-73.858255,40.726852,CRD,33.0,0.0,0.5,6.6,0.0,40.1,"{'type': 'Point', 'coordinates': [-73.963672, ...","{'type': 'Point', 'coordinates': [-73.858255, ...",NaN
3759340,CMT,2013-05-08T10:38:16.000,2013-05-08T10:41:44.000,1,0.50,-73.970260,40.762724,1,-73.963443,40.764937,CSH,4.5,0.0,0.5,0.0,0.0,5.0,"{'type': 'Point', 'coordinates': [-73.97026, 4...","{'type': 'Point', 'coordinates': [-73.963443, ...",N


In [6]:
df_trips.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,3B1A31779BCE30367D00C6F7911573C0,AED0496C937E41C4515D64E851F873AB,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:12:00,1,480,1.34,-73.982285,40.772816,-73.986214,40.758743
1,61F54249450649B22FCF456774A2F24F,9D871F2AE5ACF24D04C00484C8ECEF90,VTS,1,NaN,2013-05-01 00:03:00,2013-05-01 00:10:00,5,420,2.60,-73.963013,40.711899,-73.991875,40.721916
2,160CA9331707228AC5BD584FDBF18B3C,18F9F1A9E76B707F7D15FC2B39E0BE33,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:10:00,2,360,1.31,-73.981781,40.724354,-73.973755,40.736893
3,8F1DBE78C521F384A55AD0C77F75545D,AC4F234E82B375187FBAF428E10824D8,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:09:00,1,240,0.82,-73.964020,40.709690,-73.950897,40.710972
4,C901A9DE8D66C4F05813EB48C50F0686,10E1D1418B5B22C82255FFC638547625,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:14:00,1,540,1.65,-73.973915,40.752789,-73.996201,40.755867


In [7]:
df_trips.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,3B1A31779BCE30367D00C6F7911573C0,AED0496C937E41C4515D64E851F873AB,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:12:00,1,480,1.34,-73.982285,40.772816,-73.986214,40.758743
1,61F54249450649B22FCF456774A2F24F,9D871F2AE5ACF24D04C00484C8ECEF90,VTS,1,NaN,2013-05-01 00:03:00,2013-05-01 00:10:00,5,420,2.60,-73.963013,40.711899,-73.991875,40.721916
2,160CA9331707228AC5BD584FDBF18B3C,18F9F1A9E76B707F7D15FC2B39E0BE33,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:10:00,2,360,1.31,-73.981781,40.724354,-73.973755,40.736893
3,8F1DBE78C521F384A55AD0C77F75545D,AC4F234E82B375187FBAF428E10824D8,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:09:00,1,240,0.82,-73.964020,40.709690,-73.950897,40.710972
4,C901A9DE8D66C4F05813EB48C50F0686,10E1D1418B5B22C82255FFC638547625,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:14:00,1,540,1.65,-73.973915,40.752789,-73.996201,40.755867


### Preprocessing Data

In [8]:
df_trips.columns

Index(['medallion', ' hack_license', ' vendor_id', ' rate_code',
       ' store_and_fwd_flag', ' pickup_datetime', ' dropoff_datetime',
       ' passenger_count', ' trip_time_in_secs', ' trip_distance',
       ' pickup_longitude', ' pickup_latitude', ' dropoff_longitude',
       ' dropoff_latitude'],
      dtype='object')

In [9]:
df_trips.columns = df_trips.columns.str.strip()
df_trips.columns

Index(['medallion', 'hack_license', 'vendor_id', 'rate_code',
       'store_and_fwd_flag', 'pickup_datetime', 'dropoff_datetime',
       'passenger_count', 'trip_time_in_secs', 'trip_distance',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude'],
      dtype='object')

In [10]:
df_trips = df_trips.dropna(subset=[
    'hack_license', 'pickup_datetime', 'dropoff_datetime', 
    'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude','trip_distance'
]).drop_duplicates(subset=[
    'hack_license', 'pickup_datetime', 'dropoff_datetime', 
    'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude','trip_distance'
], keep='first')

In [11]:
df_trips = df_trips[
    (df_trips['pickup_longitude'] != 0) & (df_trips['pickup_latitude'] != 0)
    & (df_trips['dropoff_longitude'] != 0) & (df_trips['dropoff_latitude'] != 0) & (df_trips['trip_distance'] != 0)
]

### Selecting Pickup Times

In [12]:
df_trips

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,3B1A31779BCE30367D00C6F7911573C0,AED0496C937E41C4515D64E851F873AB,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:12:00,1,480,1.34,-73.982285,40.772816,-73.986214,40.758743
1,61F54249450649B22FCF456774A2F24F,9D871F2AE5ACF24D04C00484C8ECEF90,VTS,1,NaN,2013-05-01 00:03:00,2013-05-01 00:10:00,5,420,2.60,-73.963013,40.711899,-73.991875,40.721916
2,160CA9331707228AC5BD584FDBF18B3C,18F9F1A9E76B707F7D15FC2B39E0BE33,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:10:00,2,360,1.31,-73.981781,40.724354,-73.973755,40.736893
3,8F1DBE78C521F384A55AD0C77F75545D,AC4F234E82B375187FBAF428E10824D8,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:09:00,1,240,0.82,-73.964020,40.709690,-73.950897,40.710972
4,C901A9DE8D66C4F05813EB48C50F0686,10E1D1418B5B22C82255FFC638547625,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:14:00,1,540,1.65,-73.973915,40.752789,-73.996201,40.755867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15285043,738A62EEE9EC371689751A864C5EF811,5A14B7ED4F5015C9C4DF045133A44C8D,CMT,1,N,2013-05-26 18:23:30,2013-05-26 18:35:00,1,690,2.50,-74.005409,40.740875,-74.005608,40.740185
15285044,F517D6595DB783CECF674A6CA044CAFC,A3826705E0C212BE0980283F15D62B12,CMT,1,N,2013-05-26 18:09:17,2013-05-26 18:19:24,1,606,1.30,-73.987587,40.770279,-73.990059,40.758984
15285046,5519F154289E990B11E4D66C20EC8F0A,FD416115858A6A0EFCC4ACBE9432BB51,CMT,1,N,2013-05-26 18:16:40,2013-05-26 18:21:09,2,268,0.50,-73.974762,40.758827,-73.982368,40.762657
15285047,81528BB1EF6808390D1347F8AF7BE747,DFCB06E748F05A46419D7F8080BC410E,CMT,1,N,2013-05-26 22:16:17,2013-05-26 22:21:29,2,311,0.60,-74.003044,40.733028,-74.001556,40.726467


In [13]:
df_trips["pickup_datetime"] = pd.to_datetime(df_trips["pickup_datetime"])
df_trips["dropoff_datetime"] = pd.to_datetime(df_trips["dropoff_datetime"])

In [14]:
df_trips.pickup_datetime.describe()

count                         14880336
mean     2013-05-16 06:05:51.540146688
min                2013-05-01 00:00:00
25%                2013-05-08 14:43:00
50%                2013-05-16 06:14:43
75%                2013-05-23 17:56:30
max                2013-05-31 23:59:59
Name: pickup_datetime, dtype: object

In [15]:
df_trips.dropoff_datetime.describe()

count                         14880336
mean     2013-05-16 06:18:58.793802496
min                2013-05-01 00:00:00
25%                2013-05-08 14:59:47
50%                2013-05-16 06:24:52
75%                2013-05-23 18:16:00
max                2013-06-06 03:14:55
Name: dropoff_datetime, dtype: object

In [16]:
df_trips = df_trips[(df_trips.pickup_datetime >= "2013-05-01") & (df_trips.pickup_datetime < "2013-05-08")]

In [18]:
df_trips

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,3B1A31779BCE30367D00C6F7911573C0,AED0496C937E41C4515D64E851F873AB,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:12:00,1,480,1.34,-73.982285,40.772816,-73.986214,40.758743
1,61F54249450649B22FCF456774A2F24F,9D871F2AE5ACF24D04C00484C8ECEF90,VTS,1,NaN,2013-05-01 00:03:00,2013-05-01 00:10:00,5,420,2.60,-73.963013,40.711899,-73.991875,40.721916
2,160CA9331707228AC5BD584FDBF18B3C,18F9F1A9E76B707F7D15FC2B39E0BE33,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:10:00,2,360,1.31,-73.981781,40.724354,-73.973755,40.736893
3,8F1DBE78C521F384A55AD0C77F75545D,AC4F234E82B375187FBAF428E10824D8,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:09:00,1,240,0.82,-73.964020,40.709690,-73.950897,40.710972
4,C901A9DE8D66C4F05813EB48C50F0686,10E1D1418B5B22C82255FFC638547625,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:14:00,1,540,1.65,-73.973915,40.752789,-73.996201,40.755867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14518115,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:11:28,2013-05-02 12:11:28,2,0,0.90,-73.996689,40.737595,-73.985382,40.732410
14536849,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:12:03,2013-05-02 12:16:16,1,2054,5.10,-73.986328,40.718121,-73.981941,40.773323
14561682,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:06:18,2013-05-02 12:07:03,1,944,4.10,-74.001694,40.757233,-73.965981,40.789974
14565253,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:15:33,2013-05-02 12:15:33,3,0,0.80,-73.963394,40.757004,-73.954330,40.764160


In [17]:
df_trips.pickup_datetime.describe()

count                          3471518
mean     2013-05-04 12:51:42.047331328
min                2013-05-01 00:00:00
25%                2013-05-02 20:07:00
50%                2013-05-04 12:39:00
75%                2013-05-06 07:50:47
max                2013-05-07 23:59:59
Name: pickup_datetime, dtype: object

### Loading Boroughs

In [20]:
pip install geopandas

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ------------ --------------------------- 5.8/19.2 MB 29.4 MB/s eta 0:00:01
   ---------------- ----------------------- 8.1/19.2 MB 19.4 MB/s eta 0:00:01
   -------------------- ------------------- 9.7/19.2 MB 15.9 MB/s eta 0:00:01
   --------------------- ------------------ 10.5/19.2 MB 12.6 MB/s eta 0:00:01
   ------------------------ --------------- 11.8/19.2 MB 11.2 MB/s eta 0:00:01
   --------------------------- ------------ 13.4/19.2 MB 10.8 MB/s eta 0:00:01
   ------------------------------- -------- 15.2/19.2 MB 10.5 MB/s eta 0:00:01
   ----------------------------------- ---- 17.0/19.2 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 19.2/19.2 MB 10.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------- ----------------------- 2.6/6.3 MB 13.7 MB/s eta 0:00:01
   ---------------------------------- ----- 5.5/6.3 MB 14.0 MB/s eta 0:00:01

In [21]:
import geopandas as gpd
from shapely.geometry import Point

In [24]:
gdf_boroughs = gpd.read_file('data/raw/nyc_taxi/nybb_25b/')
print("Boroughs CRS:", gdf_boroughs.crs)
gdf_boroughs

Boroughs CRS: EPSG:2263


,BoroCode,BoroName,Shape_Leng,Shape_Area,geometry
0,5,Staten Island,325912.288988,1.623618e+09,"MULTIPOLYGON (((970217.022 145643.332, 970227...."
1,3,Brooklyn,728187.107889,1.934220e+09,"MULTIPOLYGON (((1022227.32 152028.146, 1022078..."
2,4,Queens,887909.387334,3.041417e+09,"MULTIPOLYGON (((1032452.015 154469.237, 103245..."
3,1,Manhattan,360037.633775,6.366460e+08,"MULTIPOLYGON (((981219.056 188655.316, 980940...."
4,2,Bronx,463147.071960,1.187199e+09,"MULTIPOLYGON (((1012821.806 229228.265, 101278..."


### Getting Boroughs

In [25]:
df_trips['pickup_geometry'] = df_trips.apply(lambda x: Point((x['pickup_longitude'], x['pickup_latitude'])), axis=1)
df_trips['dropoff_geometry'] = df_trips.apply(lambda x: Point((x['dropoff_longitude'], x['dropoff_latitude'])), axis=1)

C:\Users\akech\AppData\Local\Temp\ipykernel_13648\970991614.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_trips['pickup_geometry'] = df_trips.apply(lambda x: Point((x['pickup_longitude'], x['pickup_latitude'])), axis=1)
C:\Users\akech\AppData\Local\Temp\ipykernel_13648\970991614.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_trips['dropoff_geometry'] = df_trips.apply(lambda x: Point((x['dropoff_longitude'], x['dropoff_latitude'])), axis=1)


In [26]:
# Convert the DataFrame to a GeoDataFrame, with the correct initial CRS (EPSG:4326)
gdf_pickups = gpd.GeoDataFrame(df_trips, geometry='pickup_geometry', crs='EPSG:4326')
gdf_dropoffs = gpd.GeoDataFrame(df_trips, geometry='dropoff_geometry', crs='EPSG:4326')
# Reproject the GeoDataFrame to the CRS of the boroughs shapefile (EPSG:2263)
gdf_pickups = gdf_pickups.to_crs(gdf_boroughs.crs)
gdf_dropoffs = gdf_dropoffs.to_crs(gdf_boroughs.crs)

In [27]:
# Perform spatial join to get the boroughs
gdf_pickups = gpd.sjoin(gdf_pickups, gdf_boroughs, how='left', predicate='intersects')
gdf_dropoffs = gpd.sjoin(gdf_dropoffs, gdf_boroughs, how='left', predicate='intersects')

In [28]:
df_trips['pickup_borough'] = gdf_pickups['BoroName']
df_trips['dropoff_borough'] = gdf_dropoffs['BoroName']

C:\Users\akech\AppData\Local\Temp\ipykernel_13648\2182178560.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_trips['pickup_borough'] = gdf_pickups['BoroName']
C:\Users\akech\AppData\Local\Temp\ipykernel_13648\2182178560.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_trips['dropoff_borough'] = gdf_dropoffs['BoroName']


In [29]:
df_trips

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,pickup_geometry,dropoff_geometry,pickup_borough,dropoff_borough
0,3B1A31779BCE30367D00C6F7911573C0,AED0496C937E41C4515D64E851F873AB,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:12:00,1,480,1.34,-73.982285,40.772816,-73.986214,40.758743,POINT (-73.982285 40.772816),POINT (-73.986214 40.758743),Manhattan,Manhattan
1,61F54249450649B22FCF456774A2F24F,9D871F2AE5ACF24D04C00484C8ECEF90,VTS,1,NaN,2013-05-01 00:03:00,2013-05-01 00:10:00,5,420,2.60,-73.963013,40.711899,-73.991875,40.721916,POINT (-73.963013 40.711899),POINT (-73.991875 40.721916),Brooklyn,Manhattan
2,160CA9331707228AC5BD584FDBF18B3C,18F9F1A9E76B707F7D15FC2B39E0BE33,VTS,1,NaN,2013-05-01 00:04:00,2013-05-01 00:10:00,2,360,1.31,-73.981781,40.724354,-73.973755,40.736893,POINT (-73.981781 40.724354),POINT (-73.973755 40.736893),Manhattan,Manhattan
3,8F1DBE78C521F384A55AD0C77F75545D,AC4F234E82B375187FBAF428E10824D8,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:09:00,1,240,0.82,-73.964020,40.709690,-73.950897,40.710972,POINT (-73.96402 40.70969),POINT (-73.950897 40.710972),Brooklyn,Brooklyn
4,C901A9DE8D66C4F05813EB48C50F0686,10E1D1418B5B22C82255FFC638547625,VTS,1,NaN,2013-05-01 00:05:00,2013-05-01 00:14:00,1,540,1.65,-73.973915,40.752789,-73.996201,40.755867,POINT (-73.973915 40.752789),POINT (-73.996201 40.755867),Manhattan,Manhattan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14518115,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:11:28,2013-05-02 12:11:28,2,0,0.90,-73.996689,40.737595,-73.985382,40.732410,POINT (-73.996689 40.737595),POINT (-73.985382 40.73241),Manhattan,Manhattan
14536849,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:12:03,2013-05-02 12:16:16,1,2054,5.10,-73.986328,40.718121,-73.981941,40.773323,POINT (-73.986328 40.718121),POINT (-73.981941 40.773323),Manhattan,Manhattan
14561682,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:06:18,2013-05-02 12:07:03,1,944,4.10,-74.001694,40.757233,-73.965981,40.789974,POINT (-74.001694 40.757233),POINT (-73.965981 40.789974),Manhattan,Manhattan
14565253,144FAAC33016FED7523040FE8E28E857,8EFCFADCCE133A8CBA00282827637D9C,CMT,1,N,2013-05-02 12:15:33,2013-05-02 12:15:33,3,0,0.80,-73.963394,40.757004,-73.954330,40.764160,POINT (-73.963394 40.757004),POINT (-73.95433 40.76416),Manhattan,Manhattan


### Merging Events

In [30]:
df_trips = df_trips[
    df_trips['pickup_borough'].notna() & (df_trips['pickup_borough'] != 'Staten Island') 
    & df_trips['dropoff_borough'].notna() & (df_trips['dropoff_borough'] != 'Staten Island')].copy()

In [31]:
df_trips['pickup_type'] = df_trips['pickup_borough'] + ' Pickup'
df_trips['dropoff_type'] = df_trips['dropoff_borough'] + ' Dropoff'

In [34]:
df_pickups = df_trips[['hack_license', 'pickup_datetime', 'pickup_type','pickup_longitude','pickup_latitude','trip_distance']]\
    .rename(columns={'pickup_datetime': 'datetime', 'pickup_type': 'type','pickup_longitude': 'longitude','pickup_latitude': 'latitude'})
df_dropoffs = df_trips[['hack_license', 'dropoff_datetime', 'dropoff_type','dropoff_longitude','dropoff_latitude','trip_distance']]\
    .rename(columns={'dropoff_datetime': 'datetime', 'dropoff_type': 'type','dropoff_longitude': 'longitude','dropoff_latitude': 'latitude'})
df_all_trips = pd.concat([df_pickups, df_dropoffs], ignore_index=True)\
    .sort_values(by=['hack_license', 'datetime'])

In [35]:
df_all_trips.head(6)

,hack_license,datetime,type,longitude,latitude,trip_distance
1334161,0002555BBE359440D6CEB34B699D3932,2013-05-01 00:08:00,Brooklyn Pickup,-73.959465,40.718071,5.7
4787183,0002555BBE359440D6CEB34B699D3932,2013-05-01 00:28:00,Brooklyn Dropoff,-73.993675,40.667759,5.7
714886,0002555BBE359440D6CEB34B699D3932,2013-05-01 01:11:00,Manhattan Pickup,-73.988571,40.719658,1.0
4167908,0002555BBE359440D6CEB34B699D3932,2013-05-01 01:18:00,Manhattan Dropoff,-73.989075,40.729382,1.0
698477,0002555BBE359440D6CEB34B699D3932,2013-05-01 01:23:00,Manhattan Pickup,-73.989059,40.726952,2.3
4151499,0002555BBE359440D6CEB34B699D3932,2013-05-01 01:33:00,Brooklyn Dropoff,-73.985626,40.697857,2.3


### Getting Sequence IDs

In [36]:
df_all_trips = df_all_trips.sort_values(by=['hack_license', 'datetime'])

In [37]:
seq_ids = []
seq_count = 0
last_time = df_all_trips.loc[0, 'datetime']
last_license = df_all_trips.loc[0, 'hack_license']
max_hours = 12

for index, row in tqdm(df_all_trips.iterrows(), total=len(df_all_trips)):
    if row["hack_license"] != last_license:
        seq_count += 1
    elif (row["datetime"] - last_time).total_seconds() / 3600 > max_hours:
        seq_count += 1
    
    seq_ids.append(seq_count)
    last_time = row["datetime"]
    last_license = row["hack_license"]

df_all_trips['seq_id'] = seq_ids

  0%|          | 0/6906044 [00:00<?, ?it/s]

In [38]:
df_all_trips['seq_id'].value_counts().describe()

count    154977.000000
mean         44.561735
std          26.697045
min           1.000000
25%          30.000000
50%          44.000000
75%          56.000000
max         786.000000
Name: count, dtype: float64

### Selecting Sequences

In [39]:
seq_counts = df_all_trips['seq_id'].value_counts()
seq_counts.describe()

count    154977.000000
mean         44.561735
std          26.697045
min           1.000000
25%          30.000000
50%          44.000000
75%          56.000000
max         786.000000
Name: count, dtype: float64

In [40]:
seq_list = seq_counts[(seq_counts >= 100) & (seq_counts <= 160)]
seq_list.describe()

count    2873.000000
mean      122.485207
std        16.660125
min       100.000000
25%       108.000000
50%       120.000000
75%       134.000000
max       160.000000
Name: count, dtype: float64

In [41]:
df_all_trips = df_all_trips[df_all_trips['seq_id'].isin(seq_list.index)]
len(df_all_trips)

351900

In [42]:
df_all_trips['type'].value_counts()

type
Manhattan Pickup     163423
Manhattan Dropoff    158050
Brooklyn Dropoff       9378
Queens Dropoff         7596
Queens Pickup          7078
Brooklyn Pickup        5364
Bronx Dropoff           926
Bronx Pickup             85
Name: count, dtype: int64

### Saving Sequences

In [43]:
df_all_trips.groupby('seq_id')['datetime'].count().describe()

count    2873.000000
mean      122.485207
std        16.660125
min       100.000000
25%       108.000000
50%       120.000000
75%       134.000000
max       160.000000
Name: datetime, dtype: float64

In [47]:
df_all_trips

,hack_license,datetime,type,longitude,latitude,trip_distance,seq_id
1608201,0011B1575B9F5398BBC0F27EA560D631,2013-05-03 05:01:34,Manhattan Pickup,-73.988564,40.731632,6.70,21
5061223,0011B1575B9F5398BBC0F27EA560D631,2013-05-03 05:18:56,Manhattan Dropoff,-73.935173,40.814205,6.70,21
1589113,0011B1575B9F5398BBC0F27EA560D631,2013-05-03 06:10:25,Manhattan Pickup,-73.998413,40.740673,16.00,21
5042135,0011B1575B9F5398BBC0F27EA560D631,2013-05-03 06:53:32,Queens Dropoff,-73.847656,40.738499,16.00,21
1586716,0011B1575B9F5398BBC0F27EA560D631,2013-05-03 07:09:37,Queens Pickup,-73.847656,40.738373,6.80,21
...,...,...,...,...,...,...,...
4081992,FFD054EB3072407DA60E637EC1CDF688,2013-05-05 01:40:00,Manhattan Dropoff,-73.937675,40.821217,5.59,154906
3452231,FFD054EB3072407DA60E637EC1CDF688,2013-05-05 01:43:00,Manhattan Pickup,-73.942879,40.821678,1.76,154906
6905253,FFD054EB3072407DA60E637EC1CDF688,2013-05-05 01:49:00,Manhattan Dropoff,-73.937012,40.804371,1.76,154906
644807,FFD054EB3072407DA60E637EC1CDF688,2013-05-05 03:51:00,Manhattan Pickup,-74.002190,40.730396,6.80,154906


In [46]:
trip_seq_splits = get_seq_splits(df=df_all_trips, seq_col='seq_id')
len(trip_seq_splits)

train: 2298 seqs, val: 287 seqs, test: 288 seqs


2873

In [50]:
def save_seqs(
    df: pd.DataFrame, seq_col: str, seq_splits: dict,
    time_col: str, time_unit: float, type_col: str,lon_col: str,lat_col: str,dis_col: str ,seq_folder: str):
    """
    Save event sequences
    """
    dim_process = df[type_col].nunique()
    type_text2id = {type_text: type_id for type_id, type_text in enumerate(df[type_col].unique())}
    type_id2text = {type_id: type_text for type_text, type_id in type_text2id.items()}
    type_id_col = f'{type_col}_id'
    df[type_id_col] = df[type_col].map(type_text2id)
    data = {'train': [], 'dev': [], 'test': []}
    print(f'type_id2text: {type_id2text}')
    
    for seq_id, group in tqdm(df.groupby(seq_col)):
        group = group.sort_values(by=time_col).reset_index()
        split = seq_splits[seq_id]
        init_time = group[time_col].min()
        pre_event_time = init_time
        event_seq = {
            'dim_process': dim_process,
            'seq_idx': len(data[split]),
            'seq_len': len(group),
            'time_since_start': [],
            'time_since_last_event': [],
            'type_event': [],
            'type_text': [],
            'longitude': [],
            'latitude': [],
            'distance': [],
        }
        
        for index, row in group.iterrows():
            event_time = pd.to_datetime(row[time_col])
            time_since_start = (event_time - init_time).total_seconds() / time_unit
            time_since_last_event = (event_time - pre_event_time).total_seconds() / time_unit
            event_seq['time_since_start'].append(time_since_start)
            event_seq['time_since_last_event'].append(time_since_last_event)
            event_seq['type_event'].append(row[type_id_col])
            event_seq['type_text'].append(row[type_col])
            event_seq['longitude'].append(row[lon_col])
            event_seq['latitude'].append(row[lat_col])
            event_seq['distance'].append(row[dis_col])
            pre_event_time = event_time
        
        data[split].append(event_seq)

    os.makedirs(seq_folder, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        json_path = f'{seq_folder}/{split}.json'
        with open(json_path, 'w') as file:
            json.dump(data[split], file, indent=4)
        print(f'{split} saved to {json_path}')

In [52]:
save_seqs(
    df=df_all_trips, seq_col='seq_id', seq_splits=trip_seq_splits,
    time_col='datetime', time_unit=60*60, type_col='type',lon_col='longitude',lat_col='latitude',dis_col='trip_distance',
    seq_folder='data/nyc_taxi',
)

type_id2text: {0: 'Manhattan Pickup', 1: 'Manhattan Dropoff', 2: 'Queens Dropoff', 3: 'Queens Pickup', 4: 'Bronx Dropoff', 5: 'Brooklyn Pickup', 6: 'Brooklyn Dropoff', 7: 'Bronx Pickup'}


  0%|          | 0/2873 [00:00<?, ?it/s]

train saved to data/nyc_taxi/train.json
dev saved to data/nyc_taxi/dev.json
test saved to data/nyc_taxi/test.json


## US Earthquakes

### Downloading Data

Download the US earthquake data from 2020-01-01 (inclusive) to 2024-01-01 (exclusive) to `data/raw/us_earthquake`.

In [4]:
from io import StringIO
from datetime import datetime, timedelta

In [5]:
def download_earthquake_data_chunk(start_time, end_time, region):
    """Download earthquake data for a given time chunk."""
    
    # USGS Earthquake API endpoint
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    
    # Query parameters
    params = {
        "format": "csv",           # Output format
        "starttime": start_time,    # Start date (YYYY-MM-DD)
        "endtime": end_time,        # End date (YYYY-MM-DD)
        # "minmagnitude": min_magnitude,  # Minimum magnitude
        # "maxmagnitude": max_magnitude,  # Maximum magnitude
        "minlatitude": region["minlatitude"],  # Min latitude of region
        "maxlatitude": region["maxlatitude"],  # Max latitude
        "minlongitude": region["minlongitude"],  # Min longitude of region
        "maxlongitude": region["maxlongitude"],  # Max longitude of region
    }
    
    # Send the request
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        print(f"Data downloaded successfully for {start_time} to {end_time}.")
        return response.content
    else:
        print(f"Failed to download data for {start_time} to {end_time}. HTTP Status Code: {response.status_code}.")
        return None

In [6]:
def download_earthquake_data(start_time, end_time, region, output_file, chunk_size):
    """Download earthquake data by splitting the request into smaller chunks."""
    
    # Convert start and end times to datetime objects
    start_date = datetime.strptime(start_time, "%Y-%m-%d")
    end_date = datetime.strptime(end_time, "%Y-%m-%d")
    
    # Initialize an empty DataFrame to store all results
    all_data = pd.DataFrame()
    
    # Loop through each month in the date range
    current_start = start_date
    while current_start < end_date:
        # Define the end of the current month
        current_end = (current_start + timedelta(days=chunk_size))
        if current_end > end_date:
            current_end = end_date
        
        # Download data for the current month
        data_chunk = download_earthquake_data_chunk(
            current_start.strftime("%Y-%m-%d"), current_end.strftime("%Y-%m-%d"), region)
        
        # If data is returned, append it to the main DataFrame
        if data_chunk:
            chunk_df = pd.read_csv(StringIO(data_chunk.decode('utf-8')))
            all_data = pd.concat([all_data, chunk_df], ignore_index=True)
        
        # Move to the next month
        current_start = current_end
    
    # Save the complete dataset to a CSV file
    all_data.to_csv(output_file, index=False)
    print(f"Data downloaded successfully and saved to {output_file}.")

In [8]:
# Parameters for the earthquake search
start_time = "2020-01-01"    # Start date
end_time = "2024-01-01"      # End date
region = {
    "minlatitude": 24.6,     # Min latitude of the region
    "maxlatitude": 50.0,     # Max latitude
    "minlongitude": -125.0,  # Min longitude
    "maxlongitude": -65.0    # Max longitude
}
output_file = "data/raw/us_earthquake/us_earthquakes3.csv"

# Download the earthquake data
download_earthquake_data(start_time, end_time, region, output_file, chunk_size=30)

Data downloaded successfully for 2020-01-01 to 2020-01-31.
Data downloaded successfully for 2020-01-31 to 2020-03-01.
Data downloaded successfully for 2020-03-01 to 2020-03-31.
Data downloaded successfully for 2020-03-31 to 2020-04-30.
Data downloaded successfully for 2020-04-30 to 2020-05-30.
Data downloaded successfully for 2020-05-30 to 2020-06-29.
Data downloaded successfully for 2020-06-29 to 2020-07-29.
Data downloaded successfully for 2020-07-29 to 2020-08-28.
Data downloaded successfully for 2020-08-28 to 2020-09-27.
Data downloaded successfully for 2020-09-27 to 2020-10-27.
Data downloaded successfully for 2020-10-27 to 2020-11-26.
Data downloaded successfully for 2020-11-26 to 2020-12-26.
Data downloaded successfully for 2020-12-26 to 2021-01-25.
Data downloaded successfully for 2021-01-25 to 2021-02-24.
Data downloaded successfully for 2021-02-24 to 2021-03-26.
Data downloaded successfully for 2021-03-26 to 2021-04-25.
Data downloaded successfully for 2021-04-25 to 2021-05-2

### Loading Data

In [10]:
df_earthquakes = pd.read_csv('data/raw/us_earthquake/us_earthquakes3.csv')

In [11]:
df_earthquakes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318770 entries, 0 to 318769
Data columns (total 22 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   time             318770 non-null  object 
 1   latitude         318770 non-null  float64
 2   longitude        318770 non-null  float64
 3   depth            318770 non-null  float64
 4   mag              318702 non-null  float64
 5   magType          318701 non-null  object 
 6   nst              314522 non-null  float64
 7   gap              318756 non-null  float64
 8   dmin             317592 non-null  float64
 9   rms              318756 non-null  float64
 10  net              318770 non-null  object 
 11  id               318770 non-null  object 
 12  updated          318770 non-null  object 
 13  place            318770 non-null  object 
 14  type             318770 non-null  object 
 15  horizontalError  249436 non-null  float64
 16  depthError       318761 non-null  floa

In [12]:
pd.to_datetime(df_earthquakes.time).describe()

count                                 318770
mean     2021-09-22 19:38:25.566545664+00:00
min         2020-01-01 00:11:06.520000+00:00
25%      2020-09-01 17:12:45.875000064+00:00
50%      2021-07-20 21:06:33.011500032+00:00
75%      2022-09-09 08:06:21.096750080+00:00
max         2023-12-31 23:56:09.140000+00:00
Name: time, dtype: object

### Preprocessing Data

In [13]:
df_earthquakes = df_earthquakes[
    (df_earthquakes["type"] == "earthquake") & (df_earthquakes["status"] == "reviewed") 
    & (df_earthquakes['magType'] == 'ml')]
df_earthquakes = df_earthquakes.dropna(subset=['time', 'latitude', 'longitude', 'mag'])\
    .drop_duplicates(subset=['time', 'latitude', 'longitude', 'mag'], keep='first')
df_earthquakes["time"] = pd.to_datetime(df_earthquakes["time"])
df_earthquakes['coordinate'] = df_earthquakes.apply(
    lambda row: (round(row['latitude']), round(row['longitude'])), axis=1)

In [14]:
df_earthquakes['coordinate'].value_counts()

coordinate
(36, -118)    25859
(34, -117)    20197
(33, -116)    18415
(38, -118)    16278
(32, -104)     8230
              ...  
(46, -108)        1
(46, -77)         1
(48, -108)        1
(32, -119)        1
(45, -105)        1
Name: count, Length: 452, dtype: int64

### Getting Sequence IDs

In [15]:
df_earthquakes = df_earthquakes.sort_values(by=["coordinate", "time"]).reset_index(drop=True)

In [16]:
seq_ids = []
seq_count = 0
last_time = df_earthquakes.loc[0, 'time']
last_coord = df_earthquakes.loc[0, 'coordinate']
max_hours = 24

for index, row in tqdm(df_earthquakes.iterrows(), total=len(df_earthquakes)):
    if row["coordinate"] != last_coord:
        seq_count += 1
    elif (row["time"] - last_time).total_seconds() / 3600 > max_hours:
        seq_count += 1
    
    seq_ids.append(seq_count)
    last_time = row["time"]
    last_coord = row["coordinate"]

df_earthquakes["seq_id"] = seq_ids

  0%|          | 0/180669 [00:00<?, ?it/s]

In [17]:
df_earthquakes.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,coordinate,seq_id
0,2022-03-16 21:16:41.547000+00:00,25.3946,-100.1989,14.2300,3.4,ml,NaN,87.0,0.828,0.76,us,us6000h57x,2022-05-27T14:55:29.040Z,"5 km SW of Santiago, Mexico",earthquake,5.100000,7.100000,0.115,10.0,reviewed,us,us,"(25, -100)",0
1,2021-06-28 05:17:49.488000+00:00,26.1210,-96.2028,10.0000,2.7,ml,NaN,233.0,2.400,0.24,us,us7000eiwr,2021-09-09T22:06:53.040Z,"96 km E of South Padre Island, Texas",earthquake,11.600000,2.000000,0.105,12.0,reviewed,us,us,"(26, -96)",1
2,2021-04-11 03:42:00.619000+00:00,28.3401,-103.3944,10.0000,3.2,ml,NaN,104.0,1.018,0.70,us,us6000e0qk,2021-06-19T21:25:01.040Z,"50 km NE of Hércules, Mexico",earthquake,1.600000,2.000000,0.115,20.0,reviewed,us,us,"(28, -103)",2
3,2020-11-30 05:55:29.645000+00:00,28.4180,-100.2390,27.1698,1.5,ml,5.0,151.0,0.800,0.30,tx,tx2020xmrs,2025-07-02T20:00:41.018Z,"12 km SE of El Indio, Texas",earthquake,4.998712,6.937260,0.100,4.0,reviewed,tx,tx,"(28, -100)",3
4,2020-12-01 05:38:52.982000+00:00,28.4040,-100.3240,2.8987,2.9,ml,17.0,136.0,0.800,0.50,tx,tx2020xoms,2025-07-02T19:58:01.163Z,"11 km NNE of Guerrero, Mexico",earthquake,5.755505,2.375032,0.200,16.0,reviewed,tx,tx,"(28, -100)",3


### Selecting Sequences

In [18]:
df_earthquakes["seq_id"].value_counts().describe()

count    25480.000000
mean         7.090620
std        185.939029
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max      20126.000000
Name: count, dtype: float64

In [19]:
earthquake_counts = df_earthquakes["seq_id"].value_counts()
earthquake_list = earthquake_counts[(earthquake_counts >= 5) & (earthquake_counts <= 30)].index
len(earthquake_list)

3070

In [20]:
df_earthquakes = df_earthquakes[df_earthquakes["seq_id"].isin(earthquake_list)]

In [21]:
df_earthquakes.groupby('seq_id')['time'].count().describe()

count    3070.000000
mean        9.812052
std         5.658310
min         5.000000
25%         6.000000
50%         8.000000
75%        12.000000
max        30.000000
Name: time, dtype: float64

### Setting Event Types

In [22]:
df_earthquakes['mag'].describe()

count    30123.000000
mean         1.145491
std          0.711164
min         -1.020000
25%          0.660000
50%          1.100000
75%          1.580000
max          5.200000
Name: mag, dtype: float64

In [23]:
df_earthquakes["type"] = "Small"
df_earthquakes.loc[df_earthquakes["mag"] >= 1, "type"] = "Medium"
df_earthquakes.loc[df_earthquakes["mag"] >= 2, "type"] = "Large"
df_earthquakes["type"].value_counts()

type
Medium    13471
Small     12969
Large      3683
Name: count, dtype: int64

### Saving Sequences

In [24]:
df_earthquakes.groupby('seq_id')['time'].count().describe()

count    3070.000000
mean        9.812052
std         5.658310
min         5.000000
25%         6.000000
50%         8.000000
75%        12.000000
max        30.000000
Name: time, dtype: float64

In [27]:
earthquake_seq_splits = get_seq_splits(df=df_earthquakes, seq_col='seq_id')
len(earthquake_seq_splits)

train: 2456 seqs, val: 307 seqs, test: 307 seqs


3070

In [28]:
df_earthquakes

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,coordinate,seq_id
72,2021-07-23 20:59:36.058000+00:00,28.531000,-98.650000,3.7960,2.20,ml,11.0,89.0,0.3000,0.10,tx,tx2021oims,2025-07-02T20:01:21.524Z,"12 km NW of Tilden, Texas",Large,1.333471,1.512266,0.100000,7.0,reviewed,tx,tx,"(29, -99)",60
73,2021-07-24 08:08:40.563000+00:00,28.524000,-98.640000,4.7700,3.10,ml,21.0,72.0,0.2000,0.20,tx,tx2021ojit,2025-07-02T20:01:27.887Z,"11 km NW of Tilden, Texas",Large,1.134063,1.392107,0.000000,10.0,reviewed,tx,tx,"(29, -99)",60
74,2021-07-24 08:15:02.507000+00:00,28.518000,-98.644000,5.2568,1.90,ml,14.0,73.0,0.2000,0.20,tx,tx2021ojiy,2025-07-02T19:59:43.457Z,"11 km NW of Tilden, Texas",Medium,1.198728,1.570664,0.000000,8.0,reviewed,tx,tx,"(29, -99)",60
75,2021-07-24 17:46:59.979000+00:00,28.544000,-98.655000,3.9685,2.20,ml,16.0,80.0,0.3000,0.20,tx,tx2021okbv,2025-07-02T20:01:41.552Z,"13 km NW of Tilden, Texas",Large,1.556873,1.368526,0.100000,10.0,reviewed,tx,tx,"(29, -99)",60
76,2021-07-24 19:50:22.935000+00:00,28.520000,-98.631000,6.7249,1.80,ml,11.0,77.0,0.2000,0.20,tx,tx2021okfx,2025-07-02T20:01:38.370Z,"10 km NW of Tilden, Texas",Medium,1.183281,1.681270,0.100000,7.0,reviewed,tx,tx,"(29, -99)",60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180581,2021-12-06 00:06:56.990000+00:00,48.715667,-119.485000,0.3600,1.27,ml,9.0,172.0,0.3715,0.27,uw,uw61803126,2021-12-06T19:56:59.240Z,"3 km WNW of Tonasket, Washington",Medium,0.780000,22.830000,0.200781,6.0,reviewed,uw,uw,"(49, -119)",25401
180582,2021-12-06 04:44:35.340000+00:00,48.702667,-119.477500,0.2200,2.36,ml,16.0,172.0,0.3576,0.09,uw,uw61803151,2022-02-12T23:06:09.040Z,"2 km W of Tonasket, Washington",Large,0.450000,10.750000,0.174363,8.0,reviewed,uw,uw,"(49, -119)",25401
180583,2021-12-06 07:14:55.560000+00:00,48.704333,-119.475500,0.3400,2.16,ml,11.0,173.0,0.3589,0.09,uw,uw61803171,2022-02-12T23:06:11.040Z,"2 km W of Tonasket, Washington",Large,0.590000,10.860000,0.272194,7.0,reviewed,uw,uw,"(49, -119)",25401
180584,2021-12-06 08:55:03.160000+00:00,48.701667,-119.492167,0.1300,2.41,ml,13.0,169.0,0.3594,0.12,uw,uw61803176,2022-02-12T23:06:12.040Z,"3 km W of Tonasket, Washington",Large,0.440000,12.980000,0.065283,7.0,reviewed,uw,uw,"(49, -119)",25401


In [29]:
def save_seqs(
    df: pd.DataFrame, seq_col: str, seq_splits: dict,
    time_col: str, time_unit: float, type_col: str,mag_col: str,depth_col: str, seq_folder: str):
    """
    Save event sequences
    """
    dim_process = df[type_col].nunique()
    type_text2id = {type_text: type_id for type_id, type_text in enumerate(df[type_col].unique())}
    type_id2text = {type_id: type_text for type_text, type_id in type_text2id.items()}
    type_id_col = f'{type_col}_id'
    df[type_id_col] = df[type_col].map(type_text2id)
    data = {'train': [], 'dev': [], 'test': []}
    print(f'type_id2text: {type_id2text}')
    
    for seq_id, group in tqdm(df.groupby(seq_col)):
        group = group.sort_values(by=time_col).reset_index()
        split = seq_splits[seq_id]
        init_time = group[time_col].min()
        pre_event_time = init_time
        event_seq = {
            'dim_process': dim_process,
            'seq_idx': len(data[split]),
            'seq_len': len(group),
            'time_since_start': [],
            'time_since_last_event': [],
            'type_event': [],
            'type_text': [],
            'magnitude':[],
            'depth':[]
        }
        
        for index, row in group.iterrows():
            event_time = pd.to_datetime(row[time_col])
            time_since_start = (event_time - init_time).total_seconds() / time_unit
            time_since_last_event = (event_time - pre_event_time).total_seconds() / time_unit
            event_seq['time_since_start'].append(time_since_start)
            event_seq['time_since_last_event'].append(time_since_last_event)
            event_seq['type_event'].append(row[type_id_col])
            event_seq['type_text'].append(row[type_col])
            event_seq['magnitude'].append(row[mag_col])
            event_seq['depth'].append(row[depth_col])
            pre_event_time = event_time
        
        data[split].append(event_seq)

    os.makedirs(seq_folder, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        json_path = f'{seq_folder}/{split}.json'
        with open(json_path, 'w') as file:
            json.dump(data[split], file, indent=4)
        print(f'{split} saved to {json_path}')

In [30]:
save_seqs(
    df=df_earthquakes, seq_col='seq_id', seq_splits=earthquake_seq_splits,
    time_col='time', time_unit=60*60*24, type_col='type',mag_col='mag',depth_col='depth',
    seq_folder='data/us_earthquake3',
)

type_id2text: {0: 'Large', 1: 'Medium', 2: 'Small'}


  0%|          | 0/3070 [00:00<?, ?it/s]

train saved to data/us_earthquake3/train.json
dev saved to data/us_earthquake3/dev.json
test saved to data/us_earthquake3/test.json


## Amazon Reviews

Download the 29 small subsets (ratings only) of [Amazon Review Data](https://nijianmo.github.io/amazon/) to the folder `data/raw/amazon_review/`. Make sure to remave `AMAZON_FASHION.csv` to `Amazon_Fashion.csv`.

### Loading Data

In [ ]:
amazon_review_folder = f"{data_folder}/raw/amazon_review"
dfs_reviews = []

for file_name in tqdm(os.listdir(amazon_review_folder)):
    if file_name.endswith('.csv'):
        file_path = os.path.join(amazon_review_folder, file_name)
        df_reviews_each = pd.read_csv(file_path, names=["item", "user", "rating", "timestamp"])
        df_reviews_each['category'] = file_name.replace('.csv', '').replace('_', ' ')
        dfs_reviews.append(df_reviews_each)

df_reviews = pd.concat(dfs_reviews, ignore_index=True)

In [ ]:
df_reviews

### Preprocessing Data

In [ ]:
df_reviews['date'] = pd.to_datetime(df_reviews['timestamp'], unit='s')

In [ ]:
df_reviews['date'].describe()

In [ ]:
df_reviews = df_reviews[("2018-01-01" <= df_reviews["date"]) & (df_reviews["date"] <= "2018-06-30")]

In [ ]:
df_reviews = df_reviews.dropna(subset=['date', 'user', 'category'])\
    .drop_duplicates(subset=['date', 'user', 'category'], keep='first')

In [ ]:
len(df_reviews)

### Selecting Categories

In [ ]:
category_review_counts = df_reviews["category"].value_counts()
category_review_counts

In [ ]:
len(category_review_counts)

In [ ]:
category_list = category_review_counts[category_review_counts >= 100000].index
len(category_list)

In [ ]:
# df_reviews = df_reviews[df_reviews["category"].isin(category_list)]
df_reviews.loc[~df_reviews["category"].isin(category_list), "category"] = "Other"
len(df_reviews)

### Selecting Users

In [ ]:
user_review_counts = df_reviews["user"].value_counts()
user_review_counts

In [ ]:
user_list = user_review_counts[(user_review_counts >= 40) & (user_review_counts <= 200)].index
len(user_list)

In [ ]:
df_reviews[df_reviews["user"].isin(user_list)]["category"].value_counts()

In [ ]:
df_reviews = df_reviews[df_reviews["user"].isin(user_list)]
len(df_reviews)

### Saving Sequences

In [ ]:
df_reviews["user"].nunique(), df_reviews["category"].nunique()

In [ ]:
df_reviews["category"].value_counts()

In [ ]:
df_reviews.groupby('user')['date'].count().describe()

In [ ]:
review_seq_splits = get_seq_splits(df=df_reviews, seq_col='user')
len(review_seq_splits)

In [ ]:
save_seqs(
    df=df_reviews, seq_col='user', seq_splits=review_seq_splits,
    time_col='date', time_unit=60*60*24*7, type_col='category',
    seq_folder=f'{data_folder}/amazon_review',
)